# CNN Baseline V2, stages 1 and 2

Implements stages 1 and 2 of the approved implementation plan `AAI3001-CNN-Baseline-V2-Stages-1-and-2-Implementation-Plan.md`.

- Stage 1 tests a fixed high-pass input representation against raw RGB.
- Stage 2 tests delaying the first spatial downsampling step.

Both stages keep the baseline eight-block residual CNN, the flattened classifier head, cross entropy loss, SGD with learning rate 0.01 and momentum 0.9, batch size 64, and a 20-epoch budget. Orientation augmentation stays disabled here because stage 5 of the V2 task owns that evaluation.

**Environment.** This notebook targets a rented GPU machine. It expects the data at `/workspace/05_Data` and writes results to `/workspace/CV_test/notebooks/v2_results`. Change `DATA_ROOT` and `RESULTS_DIR` in the configuration cell if the uploaded folder layout differs.

**Run order.** Run from top to bottom. Early cells gate later ones. If a pilot records non-finite values, the dependent full-data run refuses to start until the gate is reviewed.

**Results.** Every experiment writes its own timestamped JSON record. Nothing is overwritten, so earlier runs stay available for comparison. After the run, copy `v2_results/` into `04_Outputs/CV_test/notebooks/` in the vault.

**Boundary.** The held-out test manifest is never loaded. Model selection uses validation data only.


## Setup and configuration

Paths, seeds, and experiment budgets live here. Both stages read these values, so a change here applies to every later cell and forces new experiment records under the record-matching rule.


In [ ]:
import hashlib
import itertools
import json
import math
import platform
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from PIL import Image
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm


In [ ]:
# Rental-machine layout. Change these two lines if the uploaded folder layout differs.
DATA_ROOT = Path("/workspace/05_Data")
RESULTS_DIR = DATA_ROOT.parent / "CV_test" / "notebooks" / "v2_results"

MANIFEST_DIR = DATA_ROOT / "manifests"
TRAIN_CSV = MANIFEST_DIR / "train_classifier.csv"
VAL_CSV = MANIFEST_DIR / "val_classifier.csv"
# test_classifier.csv is deliberately not loaded anywhere in this notebook.

SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 64
# The approved plan specified the baseline three-epoch budget. The budget is 20 epochs so the arms have
# room to separate. Every arm in a comparison shares this value, including the raw control, so the
# comparison stays matched. Training-duration tuning itself remains stage 4 work.
NUM_EPOCHS = 20
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NUM_WORKERS = 4

# Pilot settings carried over from the completed follow-up diagnostic so the pilot stays comparable.
PILOT_SIZES = (1, 4, 16)
PILOT_MAX_STEPS = 200
PILOT_CONFIRMATION_CHECKS = 3
PILOT_TARGET_LOSS = 0.1
LOSS_SPIKE_THRESHOLD = 10.0

# Decision rules shared by both stages. The validation split is balanced, so accuracy near 0.5 with a
# single predicted class means no learning. Two arms at that level make the stage inconclusive. A change
# is kept only when it is above chance and improves the selected-checkpoint validation accuracy by at
# least DECISION_MIN_VALIDATION_GAIN over a control that has also learned something.
NEAR_CHANCE_MARGIN = 0.02
DECISION_MIN_VALIDATION_GAIN = 0.01

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for location in (DATA_ROOT, MANIFEST_DIR, TRAIN_CSV, VAL_CSV):
    print(("OK      " if location.exists() else "MISSING ") + str(location))
print("Results directory:", RESULTS_DIR)


In [ ]:
def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def run_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def file_digest(path):
    """Hash a file so a saved record can be tied to the exact manifest it used."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def settings_digest(settings):
    payload = json.dumps(settings, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:12]


def save_json(filename, payload):
    """Write one record. Plain names land in the results folder, paths are used as given."""
    path = Path(filename)
    if not path.is_absolute():
        path = RESULTS_DIR / path
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    print("Saved", path)
    return path


def load_json(filename):
    with open(RESULTS_DIR / filename, "r", encoding="utf-8") as handle:
        return json.load(handle)


def print_table(rows, columns=None):
    frame = pd.DataFrame(rows)
    if columns is not None:
        frame = frame[[column for column in columns if column in frame.columns]]
    print(frame.to_string(index=False))


def find_experiment_record(prefixes, settings):
    """Return the newest valid saved record whose stored settings match exactly.

    A record that stopped on a non-finite loss or gradient is never reused, because its weights
    and its metrics are invalid. The settings include the manifest digests, the pilot source IDs,
    and the transform signature, so changed data or a changed filter cannot match an old record.
    """
    candidates = []
    for prefix in prefixes:
        for path in sorted(RESULTS_DIR.glob(f"{prefix}*.json")):
            try:
                payload = load_json(path.name)
            except json.JSONDecodeError:
                continue
            record = payload.get("result")
            if not isinstance(record, dict):
                continue
            if record.get("status") != "ok":
                continue
            if record.get("settings") == settings:
                candidates.append((path.name, record))
    if not candidates:
        return None
    filename, record = max(candidates, key=lambda item: item[0])
    return {"filename": filename, "record": record}


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENVIRONMENT = {
    "recorded_at": now_iso(),
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cudnn_version": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    "device": str(DEVICE),
    "determinism_note": "Seeds are fixed per run. GPU kernels are not guaranteed to reproduce bit-identical results.",
}

RUN_CONFIG = {
    "notebook": "cnn_baselineV2.ipynb",
    "stages": ["stage1_fixed_high_pass_input", "stage2_delayed_first_downsampling"],
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "optimiser": f"SGD, learning rate {LEARNING_RATE}, momentum {MOMENTUM}",
    "loss": "cross entropy",
    "num_workers": NUM_WORKERS,
    "augmentation": "disabled for stages 1 and 2, reserved for stage 5",
    "architecture": "baseline residual CNN, eight residual blocks, flatten to Linear(64*32*32, 64) then Linear(64, 2)",
    "checkpoint_rule": "best validation accuracy, ties broken by lower validation loss",
    "data_root": str(DATA_ROOT),
    "train_manifest": str(TRAIN_CSV),
    "val_manifest": str(VAL_CSV),
    "test_manifest_loaded": False,
    "pilot": {
        "sizes": list(PILOT_SIZES),
        "max_steps": PILOT_MAX_STEPS,
        "memorisation_criterion": (
            f"evaluation accuracy 1.0 and evaluation cross-entropy below {PILOT_TARGET_LOSS} "
            f"for {PILOT_CONFIRMATION_CHECKS} consecutive checks"
        ),
        "loss_spike_threshold": LOSS_SPIKE_THRESHOLD,
    },
}

print(json.dumps(ENVIRONMENT, indent=2))
save_json("stage0_environment.json", {"environment": ENVIRONMENT, "run_config": RUN_CONFIG})


## Shared definitions

The dataset, transforms, model, and experiment helpers are shared by both stages. The residual CNN keeps the baseline architecture, with one switch added for the stage 2 downsampling test.


In [ ]:
class LSBClassificationDataset(Dataset):
    """Reads one manifest row per item, decodes the PNG as RGB, and applies the transform."""

    def __init__(self, dataframe, data_root, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.data_root / str(row["image_path"])
        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")
        label = int(row["label"])
        if self.transform is not None:
            image = self.transform(image)
        return image, label


In [ ]:
LAPLACIAN_KERNEL = torch.tensor([[0.0, -1.0, 0.0], [-1.0, 4.0, -1.0], [0.0, -1.0, 0.0]])


class HighPassResidual(nn.Module):
    """Fixed per-channel Laplacian filter with reflection padding.

    The filter suppresses smooth image content and keeps high-frequency detail. Values are not
    clipped or rescaled, so the representation stays a linear function of the input tensor.
    """

    def __init__(self, kernel=None, channels=3):
        super().__init__()
        kernel = LAPLACIAN_KERNEL if kernel is None else kernel
        filters = kernel.view(1, 1, 3, 3).repeat(channels, 1, 1, 1)
        self.register_buffer("filters", filters)

    def forward(self, image):
        filters = self.filters.to(dtype=image.dtype, device=image.device)
        padded = F.pad(image.unsqueeze(0), (1, 1, 1, 1), mode="reflect")
        residual = F.conv2d(padded, filters, groups=filters.shape[0])
        return residual.squeeze(0)


raw_transform = transforms.Compose([transforms.ToTensor()])
residual_transform = transforms.Compose([transforms.ToTensor(), HighPassResidual()])

INPUT_TRANSFORMS = {"raw": raw_transform, "residual": residual_transform}


def describe_transform(transform):
    names = [type(step).__name__ for step in getattr(transform, "transforms", [])]
    return " -> ".join(names) if names else type(transform).__name__


def transform_signature(transform):
    """Record the transform settings that change the model input.

    A saved record only matches a new run when this signature matches, so a changed kernel or
    padding forces a new run instead of silently reusing an old result.
    """
    signature = []
    for step in getattr(transform, "transforms", []):
        entry = {"name": type(step).__name__}
        if isinstance(step, HighPassResidual):
            entry["kernel"] = [[float(value) for value in row] for row in step.filters[0, 0].tolist()]
            entry["padding"] = "reflect"
            entry["channels"] = int(step.filters.shape[0])
        signature.append(entry)
    return signature


In [ ]:
class ResidualBlock(nn.Module):
    """Same residual block as the baseline notebook."""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + self.shortcut(x)
        out = self.relu(out)
        return out


class ResidualSteganalysisCNN(nn.Module):
    """Baseline residual CNN with a switch for the first downsampling location.

    The eight blocks are held in a ModuleList in the baseline order. This changes parameter names
    only. The architecture, layer sizes, and parameter count match the baseline notebook.
    """

    def __init__(self, delay_first_downsampling=False):
        super().__init__()
        block4_stride = 1 if delay_first_downsampling else 2
        block5_stride = 2 if delay_first_downsampling else 1
        self.blocks = nn.ModuleList([
            ResidualBlock(3, 16, stride=1),
            ResidualBlock(16, 16, stride=1),
            ResidualBlock(16, 16, stride=1),
            ResidualBlock(16, 32, stride=block4_stride),
            ResidualBlock(32, 32, stride=block5_stride),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64, stride=1),
            ResidualBlock(64, 64, stride=1),
        ])
        self.fc1 = nn.Linear(64 * 32 * 32, 64)
        self.fc2 = nn.Linear(64, 2)
        self.relu = nn.ReLU()

    def forward_features(self, images):
        for block in self.blocks:
            images = block(images)
        return images

    def forward(self, images):
        features = self.forward_features(images)
        flattened = features.view(features.size(0), -1)

        out = self.fc1(flattened)
        out = self.relu(out)
        out = self.fc2(out)
        return out


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def classification_metrics(labels, predictions):
    labels = torch.as_tensor(labels).view(-1).long()
    predictions = torch.as_tensor(predictions).view(-1).long()
    matrix = torch.bincount(2 * labels + predictions, minlength=4).reshape(2, 2)
    true_clean, false_stego = int(matrix[0, 0]), int(matrix[0, 1])
    false_clean, true_stego = int(matrix[1, 0]), int(matrix[1, 1])
    precision = true_stego / (true_stego + false_stego) if true_stego + false_stego else 0.0
    recall = true_stego / (true_stego + false_clean) if true_stego + false_clean else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "clean_rows": int((labels == 0).sum()),
        "stego_rows": int((labels == 1).sum()),
        "predicted_clean": int((predictions == 0).sum()),
        "predicted_stego": int((predictions == 1).sum()),
        "confusion_matrix_rows_actual_clean_stego": [[true_clean, false_stego], [false_clean, true_stego]],
        "precision_stego": round(precision, 6),
        "recall_stego": round(recall, 6),
        "f1_stego": round(f1, 6),
    }


def gradients_are_finite(model):
    """Report whether every gradient is usable, so an invalid update can be refused."""
    for parameter in model.parameters():
        if parameter.grad is None:
            continue
        if not bool(torch.isfinite(parameter.grad).all().item()):
            return False
    return True


def train_one_epoch(model, loader, criterion, optimizer, device, description):
    """Run one training epoch.

    A non-finite loss or a non-finite gradient stops the epoch before the optimiser step, so a
    broken update cannot corrupt the weights and the failure is recorded instead of hidden.
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    batches_completed = 0
    nonfinite_batches = []
    spike_batch_count = 0
    spike_batch_examples = []
    stop_reason = "epoch_complete"
    stopped_early = False
    progress = tqdm(loader, desc=description, leave=False)

    for batch_index, (images, labels) in enumerate(progress, start=1):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        batch_loss = float(loss.item())

        if not math.isfinite(batch_loss):
            nonfinite_batches.append(batch_index)
            optimizer.zero_grad()
            stop_reason = "nonfinite_loss"
            stopped_early = True
            break

        if batch_loss > LOSS_SPIKE_THRESHOLD:
            spike_batch_count += 1
            if len(spike_batch_examples) < 5:
                spike_batch_examples.append([batch_index, round(batch_loss, 4)])

        loss.backward()

        if not gradients_are_finite(model):
            nonfinite_batches.append(batch_index)
            optimizer.zero_grad()
            stop_reason = "nonfinite_gradient"
            stopped_early = True
            break

        optimizer.step()
        batches_completed += 1
        total_loss += batch_loss * images.size(0)
        correct += int((outputs.argmax(dim=1) == labels).sum().item())
        total += labels.size(0)
        progress.set_postfix(loss=f"{total_loss / max(total, 1):.4f}", accuracy=f"{correct / max(total, 1):.4f}")

    return {
        "loss": None if total == 0 else total_loss / total,
        "accuracy": None if total == 0 else correct / total,
        "batches_completed": batches_completed,
        "stop_reason": stop_reason,
        "stopped_early": stopped_early,
        "nonfinite_batches": nonfinite_batches,
        "spike_batch_count": spike_batch_count,
        "spike_batch_examples": spike_batch_examples,
    }


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)
            total_loss += float(loss.item()) * labels.size(0)
            correct += int((predictions == labels).sum().item())
            total += labels.size(0)
            all_labels.append(labels.cpu())
            all_predictions.append(predictions.cpu())
    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "labels": torch.cat(all_labels),
        "predictions": torch.cat(all_predictions),
    }


In [ ]:
NOTEBOOK_VERSION = "2"
NONFINITE_STOP_REASONS = ("nonfinite_loss", "nonfinite_gradient", "nonfinite_eval_loss")


def optional_round(value, digits=4):
    return None if value is None else round(float(value), digits)


def history_entry(record, epoch):
    for entry in record.get("history", []):
        if entry.get("epoch") == epoch:
            return entry
    return {}


def pilot_settings(transform, model_kwargs, architecture):
    return {
        "notebook_version": NOTEBOOK_VERSION,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "sizes": list(PILOT_SIZES),
        "max_steps": PILOT_MAX_STEPS,
        "confirmation_checks": PILOT_CONFIRMATION_CHECKS,
        "target_loss": PILOT_TARGET_LOSS,
        "loss_spike_threshold": LOSS_SPIKE_THRESHOLD,
        "transform": describe_transform(transform),
        "transform_signature": transform_signature(transform),
        "model_kwargs": dict(model_kwargs or {}),
        "architecture": architecture,
        "pilot_source_ids": list(PILOT_SOURCE_IDS[:max(PILOT_SIZES)]),
        "data_identity": DATA_IDENTITY,
    }


def full_run_settings(transform, model_kwargs, architecture):
    return {
        "notebook_version": NOTEBOOK_VERSION,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "num_workers": NUM_WORKERS,
        "transform": describe_transform(transform),
        "transform_signature": transform_signature(transform),
        "model_kwargs": dict(model_kwargs or {}),
        "architecture": architecture,
        "data_identity": DATA_IDENTITY,
    }


def training_paths(arm_name, settings):
    """Checkpoint and progress files for one arm, keyed by the exact settings it ran with."""
    stem = f"{arm_name}__{settings_digest(settings)}"
    return (
        RESULTS_DIR / "checkpoints" / f"{stem}.pt",
        RESULTS_DIR / "progress" / f"{stem}.json",
    )


def load_training_checkpoint(path, settings, model, optimizer):
    """Restore an interrupted run when the stored settings match exactly."""
    if not path.is_file():
        return None
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if checkpoint.get("settings") != settings:
        print("A checkpoint exists for different settings. It is left untouched and the arm starts fresh.")
        return None
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    print(f"Resuming from checkpoint at epoch {checkpoint['epoch']} of {NUM_EPOCHS}.")
    return checkpoint


def save_training_checkpoint(path, epoch, model, optimizer, best, history, settings):
    """Save progress after every epoch so a disconnect does not lose the run."""
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state": {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()},
        "optimizer_state": optimizer.state_dict(),
        "best": best,
        "history": history,
        "settings": settings,
    }, path)


def run_paired_pilot(arm_name, transform, model_kwargs=None, architecture="baseline stride pattern",
                     subset_sizes=PILOT_SIZES, max_steps=PILOT_MAX_STEPS):
    """Train a fresh model on each paired subset and record per-step train and evaluation behaviour."""
    model_kwargs = dict(model_kwargs or {})
    criterion = nn.CrossEntropyLoss()
    record = {
        "arm": arm_name,
        "status": "ok",
        "transform": describe_transform(transform),
        "architecture": architecture,
        "model_kwargs": model_kwargs,
        "settings": pilot_settings(transform, model_kwargs, architecture),
        "started_at": now_iso(),
        "finished_at": None,
        "loss_spike_threshold": LOSS_SPIKE_THRESHOLD,
        "memorisation_criterion": (
            f"evaluation accuracy 1.0 and evaluation cross-entropy below {PILOT_TARGET_LOSS} "
            f"for {PILOT_CONFIRMATION_CHECKS} consecutive checks"
        ),
        "metric_timing": (
            "training metrics come from the forward pass before that step's weight update, "
            "evaluation metrics are measured with model.eval() after the update"
        ),
        "stages": [],
    }
    for size in subset_sizes:
        subset = pilot_subsets[size]
        if len(subset) == 0:
            raise RuntimeError(f"Pilot subset for {size} sources is empty.")
        seed_everything(SEED)
        dataset = LSBClassificationDataset(subset, DATA_ROOT, transform)
        loader = DataLoader(dataset, batch_size=min(BATCH_SIZE, len(subset)), shuffle=False, num_workers=0)
        model = ResidualSteganalysisCNN(**model_kwargs).to(DEVICE)
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

        history = []
        step = 0
        consecutive = 0
        stop_reason = "step_cap_reached"
        spikes = []
        nonfinite = []
        best_accuracy = {"value": None, "step": None}
        best_loss = {"value": None, "step": None}
        final_train = {"loss": None, "accuracy": None}
        final_eval = None

        for images, labels in itertools.cycle(loader):
            step += 1
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            model.train()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            train_loss = float(loss.item())
            train_accuracy = float((outputs.argmax(dim=1) == labels).float().mean().item())
            final_train = {"loss": train_loss, "accuracy": train_accuracy}

            if not math.isfinite(train_loss):
                nonfinite.append(step)
                optimizer.zero_grad()
                stop_reason = "nonfinite_loss"
                break

            loss.backward()

            if not gradients_are_finite(model):
                nonfinite.append(step)
                optimizer.zero_grad()
                stop_reason = "nonfinite_gradient"
                break

            optimizer.step()

            evaluation = evaluate(model, loader, criterion, DEVICE)
            eval_loss = float(evaluation["loss"])
            eval_accuracy = float(evaluation["accuracy"])
            final_eval = evaluation

            if not math.isfinite(eval_loss):
                nonfinite.append(step)
                stop_reason = "nonfinite_eval_loss"
                break

            if train_loss > LOSS_SPIKE_THRESHOLD:
                spikes.append(step)

            if best_accuracy["value"] is None or eval_accuracy > best_accuracy["value"]:
                best_accuracy = {"value": eval_accuracy, "step": step}
            if best_loss["value"] is None or eval_loss < best_loss["value"]:
                best_loss = {"value": eval_loss, "step": step}

            consecutive = consecutive + 1 if (eval_accuracy >= 1.0 and eval_loss < PILOT_TARGET_LOSS) else 0
            if consecutive >= PILOT_CONFIRMATION_CHECKS:
                stop_reason = "memorisation_criterion_met"

            history.append({
                "step": step,
                "train_loss": train_loss,
                "train_accuracy": train_accuracy,
                "eval_loss": eval_loss,
                "eval_accuracy": eval_accuracy,
                "eval_predicted_clean": int((evaluation["predictions"] == 0).sum()),
                "eval_predicted_stego": int((evaluation["predictions"] == 1).sum()),
            })

            if stop_reason == "memorisation_criterion_met" or step >= max_steps:
                break

        record["stages"].append({
            "sources": size,
            "rows": int(len(subset)),
            "source_ids": sorted(subset["source_id"].astype(str).unique().tolist()),
            "payload_rates": {
                str(row.source_id): round(float(row.payload_rate), 6)
                for row in subset.loc[subset["label"] == 1, ["source_id", "payload_rate"]].itertuples(index=False)
            },
            "steps_run": step,
            "stop_reason": stop_reason,
            "invalid_training": stop_reason in NONFINITE_STOP_REASONS,
            "memorised": stop_reason == "memorisation_criterion_met",
            "final_train_loss": final_train["loss"],
            "final_train_accuracy": final_train["accuracy"],
            "final_eval_loss": None if final_eval is None else float(final_eval["loss"]),
            "final_eval_accuracy": None if final_eval is None else float(final_eval["accuracy"]),
            "final_eval_metrics": (
                None if final_eval is None
                else classification_metrics(final_eval["labels"], final_eval["predictions"])
            ),
            "best_eval_accuracy": best_accuracy["value"],
            "best_eval_accuracy_step": best_accuracy["step"],
            "best_eval_loss": best_loss["value"],
            "best_eval_loss_step": best_loss["step"],
            "loss_spike_steps": spikes,
            "nonfinite_steps": sorted(set(nonfinite)),
            "history": history,
        })
        stage = record["stages"][-1]
        if stage["invalid_training"]:
            print(f"{arm_name} | {size} sources | stopped at step {step} with {stop_reason}. The stage is invalid.")
            record["status"] = f"stopped_{stop_reason}"
        else:
            print(f"{arm_name} | {size} sources | {step} steps | "
                  f"final train accuracy {stage['final_train_accuracy']:.4f} | "
                  f"final eval accuracy {stage['final_eval_accuracy']:.4f} | "
                  f"memorised {stage['memorised']}")

    record["finished_at"] = now_iso()
    return record


def run_full_experiment(arm_name, transform, model_kwargs=None, architecture="baseline stride pattern"):
    """Train one arm on the full training split and select the checkpoint on validation accuracy.

    Progress and the best weights are written after every epoch, so an interrupted rental session
    resumes from its checkpoint instead of starting over. An arm that hits a non-finite loss or
    gradient stops, is marked invalid, and keeps no checkpoint.

    A resumed arm reshuffles from the fixed seed, so its epoch order can differ from an
    uninterrupted run. The difference is reported in the record as resumed_from_epoch and it does
    not bias the comparison, because both arms are trained the same way per epoch index.
    """
    model_kwargs = dict(model_kwargs or {})
    settings = full_run_settings(transform, model_kwargs, architecture)
    checkpoint_file, progress_file = training_paths(arm_name, settings)

    seed_everything(SEED)
    train_loader = DataLoader(
        LSBClassificationDataset(train_df, DATA_ROOT, transform),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        LSBClassificationDataset(val_df, DATA_ROOT, transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    )
    criterion = nn.CrossEntropyLoss()
    model = ResidualSteganalysisCNN(**model_kwargs).to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    history = []
    best = {"epoch": None, "val_accuracy": None, "val_loss": None, "state_dict": None}
    resumed_from_epoch = 0
    status = "ok"
    stop_reason = "completed_all_epochs"
    started_at = now_iso()
    started = time.time()

    checkpoint = load_training_checkpoint(checkpoint_file, settings, model, optimizer)
    if checkpoint is None:
        start_epoch = 1
    else:
        history = checkpoint["history"]
        best = checkpoint["best"]
        resumed_from_epoch = int(checkpoint["epoch"])
        start_epoch = resumed_from_epoch + 1
        if start_epoch > NUM_EPOCHS:
            print("The checkpoint already covers every epoch. Rebuilding the final record from it.")

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        training = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE,
                                   description=f"{arm_name} epoch {epoch}/{NUM_EPOCHS}")
        entry = {
            "epoch": epoch,
            "train_loss": training["loss"],
            "train_accuracy": training["accuracy"],
            "train_batches_completed": training["batches_completed"],
            "train_stop_reason": training["stop_reason"],
            "train_stopped_early": training["stopped_early"],
            "train_nonfinite_batches": training["nonfinite_batches"],
            "train_spike_batch_count": training["spike_batch_count"],
            "train_spike_batch_examples": training["spike_batch_examples"],
            "val_loss": None,
            "val_accuracy": None,
            "val_metrics": None,
            "elapsed_seconds": round(time.time() - started, 1),
        }

        if training["stopped_early"]:
            history.append(entry)
            status = f"stopped_{training['stop_reason']}"
            stop_reason = training["stop_reason"]
            print(f"epoch {epoch} stopped early with {training['stop_reason']} at batch "
                  f"{training['nonfinite_batches'][-1]}. The arm is invalid and no checkpoint is kept.")
            break

        evaluation = evaluate(model, val_loader, criterion, DEVICE)
        val_accuracy = float(evaluation["accuracy"])
        val_loss = float(evaluation["loss"])
        metrics = classification_metrics(evaluation["labels"], evaluation["predictions"])
        entry["val_loss"] = val_loss
        entry["val_accuracy"] = val_accuracy
        entry["val_metrics"] = metrics
        history.append(entry)

        if best["val_accuracy"] is None or val_accuracy > best["val_accuracy"] or (
                val_accuracy == best["val_accuracy"] and val_loss < best["val_loss"]):
            best = {
                "epoch": epoch,
                "val_accuracy": val_accuracy,
                "val_loss": val_loss,
                "state_dict": {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()},
            }

        save_training_checkpoint(checkpoint_file, epoch, model, optimizer, best, history, settings)
        save_json(progress_file, {
            "updated_at": now_iso(),
            "arm": arm_name,
            "settings": settings,
            "epochs_completed": epoch,
            "epochs_planned": NUM_EPOCHS,
            "best_epoch": best["epoch"],
            "best_val_accuracy": best["val_accuracy"],
            "history": history,
        })
        print(f"epoch {epoch} | train loss {training['loss']:.4f} | train accuracy {training['accuracy']:.4f} | "
              f"val loss {val_loss:.4f} | val accuracy {val_accuracy:.4f} | "
              f"predicted clean {metrics['predicted_clean']} | predicted stego {metrics['predicted_stego']}")

    if status != "ok" or best["epoch"] is None:
        selected_metrics = None
        selected_epoch = best["epoch"]
        if checkpoint_file.is_file():
            checkpoint_file.unlink()
            print(f"Removed the invalid checkpoint {checkpoint_file.name}.")
    else:
        model.load_state_dict(best["state_dict"])
        selected = evaluate(model, val_loader, criterion, DEVICE)
        selected_metrics = {
            "val_loss": float(selected["loss"]),
            "val_accuracy": float(selected["accuracy"]),
            **classification_metrics(selected["labels"], selected["predictions"]),
        }
        selected_epoch = best["epoch"]

    return {
        "arm": arm_name,
        "status": status,
        "stop_reason": stop_reason,
        "transform": describe_transform(transform),
        "architecture": architecture,
        "model_kwargs": model_kwargs,
        "settings": settings,
        "started_at": started_at,
        "finished_at": now_iso(),
        "epochs": NUM_EPOCHS,
        "epochs_completed": history[-1]["epoch"] if history else 0,
        "resumed_from_epoch": resumed_from_epoch,
        "history": history,
        "selected_epoch": selected_epoch,
        "selected_checkpoint_metrics": selected_metrics,
        "checkpoint_file": checkpoint_file.name if checkpoint_file.is_file() else None,
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None,
        "elapsed_seconds": round(time.time() - started, 1),
    }


def run_or_reuse_pilot(arm_name, transform, model_kwargs, architecture, prefix):
    settings = pilot_settings(transform, model_kwargs, architecture)
    match = find_experiment_record(("stage1_pilot_", "stage2_pilot_"), settings)
    if match is not None:
        print(f"Reusing saved pilot record {match['filename']} for {arm_name}")
        return match["record"], match["filename"], True
    record = run_paired_pilot(arm_name, transform, model_kwargs=model_kwargs, architecture=architecture)
    filename = f"{prefix}{arm_name}__{run_stamp()}.json"
    save_json(filename, {"saved_at": now_iso(), "run_config": RUN_CONFIG, "result": record})
    if record["status"] != "ok":
        print(f"{arm_name} ended with status {record['status']}. The record is saved and will not be reused.")
    return record, filename, False


def run_or_reuse_full(prefix, arm_name, transform, model_kwargs, architecture):
    settings = full_run_settings(transform, model_kwargs, architecture)
    match = find_experiment_record(("stage1_full_", "stage2_full_"), settings)
    if match is not None:
        print(f"Reusing saved full-run record {match['filename']} for {arm_name}")
        return match["record"], match["filename"], True
    record = run_full_experiment(arm_name, transform, model_kwargs=model_kwargs, architecture=architecture)
    filename = f"{prefix}{arm_name}__{run_stamp()}.json"
    save_json(filename, {"saved_at": now_iso(), "run_config": RUN_CONFIG, "result": record})
    if record["status"] != "ok":
        print(f"{arm_name} ended with status {record['status']}. The record is saved and will not be reused.")
    return record, filename, False


def evaluate_pilot_gate(records):
    rows = []
    nonfinite_any = False
    spike_any = False
    invalid_any = False
    for arm_name, record in records.items():
        for stage in record["stages"]:
            nonfinite_any = nonfinite_any or bool(stage["nonfinite_steps"])
            spike_any = spike_any or bool(stage["loss_spike_steps"])
            invalid_any = invalid_any or bool(stage.get("invalid_training"))
            rows.append({
                "arm": arm_name,
                "sources": stage["sources"],
                "steps": stage["steps_run"],
                "stop reason": stage["stop_reason"],
                "final train accuracy": optional_round(stage["final_train_accuracy"]),
                "final eval accuracy": optional_round(stage["final_eval_accuracy"]),
                "final eval loss": optional_round(stage["final_eval_loss"]),
                "best eval accuracy": optional_round(stage["best_eval_accuracy"]),
                "memorised": stage["memorised"],
                "spike steps": len(stage["loss_spike_steps"]),
                "nonfinite steps": len(stage["nonfinite_steps"]),
            })
    verdict = "stop" if (nonfinite_any or invalid_any) else ("review_spikes" if spike_any else "proceed")
    return {
        "rows": rows,
        "nonfinite_any": nonfinite_any,
        "spike_any": spike_any,
        "invalid_any": invalid_any,
        "verdict": verdict,
    }


def result_is_informative(record):
    """Describe whether a validation result shows any learning.

    The validation split is balanced, so accuracy close to 0.5 with a single predicted class
    means the model has not learned the task.
    """
    selected = record.get("selected_checkpoint_metrics")
    if not selected:
        return {"accuracy": None, "collapsed": None, "above_chance": False, "informative": False}
    accuracy = float(selected["val_accuracy"])
    collapsed = selected["predicted_clean"] == 0 or selected["predicted_stego"] == 0
    above_chance = accuracy >= 0.5 + NEAR_CHANCE_MARGIN
    return {
        "accuracy": accuracy,
        "collapsed": bool(collapsed),
        "above_chance": bool(above_chance),
        "informative": bool(above_chance and not collapsed),
    }


def stage_decision(control_record, candidate_record, keep_label, control_label):
    """Compare two arms without treating a chance-level result as an improvement."""
    control = result_is_informative(control_record)
    candidate = result_is_informative(candidate_record)
    gain = None
    if control["accuracy"] is not None and candidate["accuracy"] is not None:
        gain = round(candidate["accuracy"] - control["accuracy"], 6)

    if control["accuracy"] is None or candidate["accuracy"] is None:
        recommendation = "inconclusive_missing_result"
    elif not control["informative"] and not candidate["informative"]:
        recommendation = "inconclusive_no_learning"
    elif candidate["informative"] and not control["informative"]:
        recommendation = keep_label
    elif gain is not None and gain >= DECISION_MIN_VALIDATION_GAIN:
        recommendation = keep_label
    else:
        recommendation = control_label

    return {
        "recommendation": recommendation,
        "gain": gain,
        "near_chance_margin": NEAR_CHANCE_MARGIN,
        "keep_threshold": DECISION_MIN_VALIDATION_GAIN,
        "control": control,
        "candidate": candidate,
    }


def full_run_row(arm_name, record):
    selected = record.get("selected_checkpoint_metrics")
    row = {
        "arm": arm_name,
        "status": record.get("status", "unknown"),
        "input": record["transform"],
        "architecture": record["architecture"],
        "selected epoch": record.get("selected_epoch"),
    }
    if not selected:
        row.update({
            "train accuracy": None,
            "train loss": None,
            "val accuracy": None,
            "val loss": None,
            "precision stego": None,
            "recall stego": None,
            "f1 stego": None,
            "predicted clean": None,
            "predicted stego": None,
            "collapsed": None,
            "informative": False,
            "final epoch val accuracy": None,
        })
        return row

    informative = result_is_informative(record)
    entry = history_entry(record, record.get("selected_epoch"))
    final_accuracy = None
    if record.get("history") and record["history"][-1].get("val_accuracy") is not None:
        final_accuracy = round(record["history"][-1]["val_accuracy"], 4)
    row.update({
        "train accuracy": optional_round(entry.get("train_accuracy")),
        "train loss": optional_round(entry.get("train_loss")),
        "val accuracy": round(selected["val_accuracy"], 4),
        "val loss": round(selected["val_loss"], 4),
        "precision stego": selected["precision_stego"],
        "recall stego": selected["recall_stego"],
        "f1 stego": selected["f1_stego"],
        "predicted clean": selected["predicted_clean"],
        "predicted stego": selected["predicted_stego"],
        "collapsed": informative["collapsed"],
        "informative": informative["informative"],
        "final epoch val accuracy": final_accuracy,
    })
    return row


## Data and training-stability preflight

These checks run before any experiment.

1. The split manifests load with string source IDs, contain only labels 0 and 1, and pair every source with exactly one clean and one stego row.
2. Source identities do not cross the train and validation splits.
3. Every referenced image file exists, and sampled images decode as 128 by 128 RGB files.
4. The raw and residual representations produce finite inputs, finite logits and loss, usable gradients, real parameter updates, and predictions in both train and evaluation mode.

A failure here stops the notebook, because a model comparison on broken inputs would be meaningless. The separate diagnostic manifests are not used, and their duplicate-row flag is recorded rather than re-checked on data this notebook does not load.


In [ ]:
for required in (TRAIN_CSV, VAL_CSV):
    if not required.is_file():
        raise FileNotFoundError(
            f"Missing manifest {required}. Upload the 05_Data folder and point DATA_ROOT at it."
        )

dtype_map = {"source_id": str, "generator_source_id": str}
train_df = pd.read_csv(TRAIN_CSV, dtype=dtype_map)
val_df = pd.read_csv(VAL_CSV, dtype=dtype_map)

# Identity of the data a record was produced from. Changing the manifests forces new runs.
DATA_IDENTITY = {
    "train_manifest": str(TRAIN_CSV),
    "train_sha256": file_digest(TRAIN_CSV),
    "val_manifest": str(VAL_CSV),
    "val_sha256": file_digest(VAL_CSV),
}
print("Train manifest digest:", DATA_IDENTITY["train_sha256"][:16])
print("Validation manifest digest:", DATA_IDENTITY["val_sha256"][:16])


def manifest_structure(frame, name):
    pairing = frame.groupby("source_id")["label"].agg(
        lambda values: (int((values == 0).sum()), int((values == 1).sum()))
    )
    patterns = {}
    for (clean_count, stego_count), count in pairing.value_counts().items():
        patterns[f"clean_{clean_count}_stego_{stego_count}"] = int(count)
    stego_rows = frame[frame["label"] == 1]
    return {
        "name": name,
        "rows": int(len(frame)),
        "columns": list(frame.columns),
        "source_count": int(frame["source_id"].nunique()),
        "clean_rows": int((frame["label"] == 0).sum()),
        "stego_rows": int(stego_rows.shape[0]),
        "unexpected_labels": sorted(int(value) for value in set(frame["label"].unique()) - {0, 1}),
        "duplicate_source_label_rows": int(frame.duplicated(subset=["source_id", "label"]).sum()),
        "pairing_patterns": patterns,
        "stego_payload_rate_min": float(stego_rows["payload_rate"].min()),
        "stego_payload_rate_max": float(stego_rows["payload_rate"].max()),
    }


train_structure = manifest_structure(train_df, "train")
val_structure = manifest_structure(val_df, "validation")
overlap = sorted(set(train_df["source_id"]) & set(val_df["source_id"]))

print_table([
    {"split": "train", "rows": train_structure["rows"], "sources": train_structure["source_count"],
     "clean": train_structure["clean_rows"], "stego": train_structure["stego_rows"],
     "duplicate rows": train_structure["duplicate_source_label_rows"]},
    {"split": "validation", "rows": val_structure["rows"], "sources": val_structure["source_count"],
     "clean": val_structure["clean_rows"], "stego": val_structure["stego_rows"],
     "duplicate rows": val_structure["duplicate_source_label_rows"]},
])
print("Pairing patterns, train:", train_structure["pairing_patterns"])
print("Pairing patterns, validation:", val_structure["pairing_patterns"])
print("Sources appearing in both train and validation:", len(overlap))


In [ ]:
def check_manifest_files(frame, data_root, description):
    missing_count = 0
    missing_examples = []
    for relative_path in tqdm(frame["image_path"].astype(str), desc=f"Checking {description}", leave=False):
        if not (data_root / relative_path).is_file():
            missing_count += 1
            if len(missing_examples) < 10:
                missing_examples.append(relative_path)
    return {"checked": int(len(frame)), "missing_count": missing_count, "missing_examples": missing_examples}


def decode_samples(frame, data_root, description, per_class=3):
    samples = []
    for label in (0, 1):
        for row in frame[frame["label"] == label].head(per_class).itertuples(index=False):
            with Image.open(data_root / str(row.image_path)) as image_file:
                samples.append({
                    "split": description,
                    "label": int(row.label),
                    "path": str(row.image_path),
                    "file_mode": image_file.mode,
                    "size": list(image_file.size),
                    "format": image_file.format,
                })
    return samples


train_files = check_manifest_files(train_df, DATA_ROOT, "train images")
val_files = check_manifest_files(val_df, DATA_ROOT, "validation images")
samples = decode_samples(train_df, DATA_ROOT, "train") + decode_samples(val_df, DATA_ROOT, "validation")

save_json("stage0_preflight.json", {
    "checked_at": now_iso(),
    "run_config": RUN_CONFIG,
    "train": train_structure,
    "validation": val_structure,
    "split_overlap_source_count": len(overlap),
    "split_overlap_examples": overlap[:10],
    "train_files": train_files,
    "validation_files": val_files,
    "decode_samples": samples,
    "test_manifest_loaded": False,
    "note": (
        "The follow-up audit flagged duplicate source rows in the separate diagnostic 1-bit and 4-bit "
        "manifests. This check covers the main split manifests used for stages 1 and 2."
    ),
})

problems = []
if train_structure["unexpected_labels"] or val_structure["unexpected_labels"]:
    problems.append("unexpected labels present")
if train_structure["duplicate_source_label_rows"] or val_structure["duplicate_source_label_rows"]:
    problems.append("duplicate (source_id, label) rows present")
if set(train_structure["pairing_patterns"]) != {"clean_1_stego_1"}:
    problems.append("train sources are not all paired as one clean and one stego row")
if set(val_structure["pairing_patterns"]) != {"clean_1_stego_1"}:
    problems.append("validation sources are not all paired as one clean and one stego row")
if len(overlap):
    problems.append("source identities cross the train and validation splits")
if train_files["missing_count"] or val_files["missing_count"]:
    problems.append("referenced image files are missing")
if any(sample["size"] != [IMAGE_SIZE, IMAGE_SIZE] for sample in samples):
    problems.append("decoded sample images do not match the expected image size")

print_table([
    {"split": "train", "checked": train_files["checked"], "missing": train_files["missing_count"]},
    {"split": "validation", "checked": val_files["checked"], "missing": val_files["missing_count"]},
])
print("Sampled image sizes and modes:", sorted({(tuple(sample["size"]), sample["file_mode"]) for sample in samples}))

if problems:
    raise RuntimeError("Preflight failed, resolve these before any experiment: " + "; ".join(problems))
print(f"Preflight passed for {train_structure['rows']} train rows and {val_structure['rows']} validation rows.")


In [ ]:
# The same 16 source IDs used in the completed paired diagnostic, kept so the pilot stays comparable.
PILOT_SOURCE_IDS = [
    "55548", "44480", "12064", "28175", "30693", "22268", "20762", "03227",
    "29019", "42688", "15513", "26180", "49072", "38087", "09693", "44821",
]

pilot_subsets = {}
subset_records = []
subset_problems = []
for size in PILOT_SIZES:
    source_ids = PILOT_SOURCE_IDS[:size]
    subset = train_df[train_df["source_id"].isin(source_ids)].copy()
    subset = subset.sort_values(["source_id", "label"], kind="mergesort").reset_index(drop=True)
    clean_rows = int((subset["label"] == 0).sum())
    stego_rows = int((subset["label"] == 1).sum())
    if len(subset) != 2 * size or clean_rows != size or stego_rows != size:
        subset_problems.append(
            f"{size} sources expected {2 * size} rows with {size} per class, found {len(subset)} rows "
            f"with {clean_rows} clean and {stego_rows} stego"
        )
    pilot_subsets[size] = subset
    subset_records.append({
        "sources": size,
        "rows": int(len(subset)),
        "clean_rows": clean_rows,
        "stego_rows": stego_rows,
        "source_ids": source_ids,
        "payload_rates": {
            str(row.source_id): round(float(row.payload_rate), 6)
            for row in subset.loc[subset["label"] == 1, ["source_id", "payload_rate"]].itertuples(index=False)
        },
    })

save_json("stage0_pilot_subsets.json", {
    "checked_at": now_iso(),
    "source_id_source": "diagnostic_results/test5_preflight.json, reused so the pilot stays comparable",
    "pilot_sizes": list(PILOT_SIZES),
    "subsets": subset_records,
})
print_table(subset_records, columns=["sources", "rows", "clean_rows", "stego_rows"])

if subset_problems:
    raise RuntimeError("Pilot subset problem: " + "; ".join(subset_problems))
print("Paired pilot subsets ready.")


In [ ]:
def verification_checks(name, transform, model_kwargs=None):
    """Check finite values, gradients, real parameter updates, and both prediction modes."""
    model_kwargs = dict(model_kwargs or {})
    subset = pilot_subsets[max(PILOT_SIZES)]
    dataset = LSBClassificationDataset(subset, DATA_ROOT, transform)
    loader = DataLoader(dataset, batch_size=len(subset), shuffle=False, num_workers=0)
    images, labels = next(iter(loader))

    result = {
        "arm": name,
        "transform": describe_transform(transform),
        "model_kwargs": model_kwargs,
        "subset_rows": int(len(subset)),
        "input": {
            "shape": list(images.shape),
            "dtype": str(images.dtype),
            "finite": bool(torch.isfinite(images).all().item()),
            "min": round(float(images.min()), 6),
            "max": round(float(images.max()), 6),
            "mean": round(float(images.mean()), 6),
        },
    }

    seed_everything(SEED)
    model = ResidualSteganalysisCNN(**model_kwargs).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

    model.train()
    optimizer.zero_grad()
    outputs = model(images.to(DEVICE))
    loss = criterion(outputs, labels.to(DEVICE))
    loss.backward()
    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    result["forward"] = {
        "logits_shape": list(outputs.shape),
        "logits_finite": bool(torch.isfinite(outputs).all().item()),
        "loss": round(float(loss.item()), 6),
        "loss_finite": bool(math.isfinite(float(loss.item()))),
    }
    result["gradients"] = {
        "tensors_with_gradients": len(gradients),
        "all_finite": bool(all(torch.isfinite(gradient).all().item() for gradient in gradients)),
        "max_abs_gradient": round(float(torch.stack([g.abs().max() for g in gradients]).max().item()), 6),
        "any_nonzero": bool(any(float(g.abs().sum().item()) > 0 for g in gradients)),
    }
    before = [parameter.detach().clone() for parameter in model.parameters()]
    optimizer.step()
    changed = sum(
        1 for previous, current in zip(before, model.parameters())
        if not torch.equal(previous, current.detach())
    )
    result["optimiser_update"] = {"parameters_changed": changed, "parameters_total": len(before)}

    with torch.no_grad():
        train_predictions = model(images.to(DEVICE)).argmax(dim=1)
    model.eval()
    with torch.no_grad():
        eval_predictions = model(images.to(DEVICE)).argmax(dim=1)
    result["train_mode_predictions"] = {
        "predicted_clean": int((train_predictions == 0).sum()),
        "predicted_stego": int((train_predictions == 1).sum()),
    }
    result["eval_mode_predictions"] = {
        "predicted_clean": int((eval_predictions == 0).sum()),
        "predicted_stego": int((eval_predictions == 1).sum()),
    }
    return result


def transform_pair_check(name, transform):
    """Check determinism, finite output, expected shape, and a nonzero clean/stego difference."""
    pair_frame = pilot_subsets[1]
    labels = pair_frame["label"].tolist()
    if labels != [0, 1]:
        raise RuntimeError(f"Expected one clean row followed by one stego row, found labels {labels}")
    dataset = LSBClassificationDataset(pair_frame, DATA_ROOT, transform)
    clean = dataset[0][0]
    stego = dataset[1][0]
    repeated = dataset[0][0]
    return {
        "arm": name,
        "transform": describe_transform(transform),
        "source_id": str(pair_frame["source_id"].iloc[0]),
        "shape": list(clean.shape),
        "finite": bool(torch.isfinite(clean).all().item() and torch.isfinite(stego).all().item()),
        "deterministic": bool(torch.equal(clean, repeated)),
        "pair_differs": bool(not torch.equal(clean, stego)),
        "pair_changed_values": int((clean != stego).sum().item()),
        "pair_max_abs_difference": round(float((clean - stego).abs().max().item()), 6),
    }


verification = [
    verification_checks("raw control", raw_transform),
    verification_checks("residual", residual_transform),
]
pair_checks = [
    transform_pair_check("raw control", raw_transform),
    transform_pair_check("residual", residual_transform),
]

save_json("stage0_verification.json", {
    "checked_at": now_iso(),
    "run_config": RUN_CONFIG,
    "verification": verification,
    "pair_checks": pair_checks,
})

print_table([
    {
        "arm": entry["arm"],
        "input min": entry["input"]["min"],
        "input max": entry["input"]["max"],
        "gradients finite": entry["gradients"]["all_finite"],
        "parameters changed": entry["optimiser_update"]["parameters_changed"],
        "train predicted clean": entry["train_mode_predictions"]["predicted_clean"],
        "train predicted stego": entry["train_mode_predictions"]["predicted_stego"],
        "eval predicted clean": entry["eval_mode_predictions"]["predicted_clean"],
        "eval predicted stego": entry["eval_mode_predictions"]["predicted_stego"],
    }
    for entry in verification
])
print_table(pair_checks, columns=["arm", "pair_changed_values", "pair_max_abs_difference", "deterministic", "pair_differs"])

problems = []
for entry in verification:
    if not entry["input"]["finite"] or not entry["forward"]["logits_finite"] or not entry["forward"]["loss_finite"]:
        problems.append(f"{entry['arm']} produced non-finite input, logits, or loss")
    if not entry["gradients"]["all_finite"] or not entry["gradients"]["any_nonzero"]:
        problems.append(f"{entry['arm']} produced missing or non-finite gradients")
    if entry["optimiser_update"]["parameters_changed"] == 0:
        problems.append(f"{entry['arm']} did not update any parameter")
    if sum(entry["train_mode_predictions"].values()) != entry["subset_rows"]:
        problems.append(f"{entry['arm']} train-mode prediction count does not match the subset size")
for entry in pair_checks:
    if not entry["deterministic"] or not entry["finite"] or not entry["pair_differs"]:
        problems.append(f"{entry['arm']} transform check failed")

if problems:
    raise RuntimeError("Verification failed: " + "; ".join(problems))
print("Verification passed for both representations.")


## Stage 1. Fixed high-pass input

**Question.** Does a fixed noise-residual representation help the unchanged CNN learn the main 1-bit task compared with raw RGB under the same training conditions?

**Change.** A fixed per-channel 3 by 3 Laplacian filter with reflection padding is applied after `ToTensor()`. Nothing else changes, so the filter is the only difference from the raw control.

**Method.** Both arms run the paired pilot at 1, 4, and 16 sources, then train on the full training split at batch size 64 for 20 epochs. Checkpoints are selected on validation accuracy.

**Safety.** A non-finite loss or gradient stops an arm before the optimiser step. That arm is marked invalid and its result is never reused.

**Gate.** Non-finite pilot values block the full-data runs. Recorded loss spikes require an explicit override after review, because the raw baseline has already shown transient spikes at these settings.

**Decision.** The residual input is kept only when it is above chance and beats the raw control by the configured margin. Two chance-level arms are reported as inconclusive, and Stage 2 is then blocked until an override is set with a recorded reason.


In [ ]:
STAGE1_ARMS = {
    "raw_control": {"transform": raw_transform, "model_kwargs": {}, "architecture": "baseline stride pattern"},
    "residual": {"transform": residual_transform, "model_kwargs": {}, "architecture": "baseline stride pattern"},
}

# Set True only after reading the stage 1 pilot gate output and judging the recorded loss spikes
# transient and harmless. Non-finite values block the full runs regardless of this flag.
ALLOW_STAGE1_FULL_RUN_AFTER_SPIKES = False

print("Stage 1 arms:", {name: describe_transform(arm["transform"]) for name, arm in STAGE1_ARMS.items()})
print("Spike override flag:", ALLOW_STAGE1_FULL_RUN_AFTER_SPIKES)


In [ ]:
stage1_pilots = {}
for arm_name, arm in STAGE1_ARMS.items():
    print("=" * 72)
    print("Stage 1 paired pilot |", arm_name)
    record, filename, reused = run_or_reuse_pilot(
        arm_name, arm["transform"], arm["model_kwargs"], arm["architecture"], "stage1_pilot_")
    stage1_pilots[arm_name] = record
    print(("Reused " if reused else "Saved ") + filename)

stage1_pilot_table = evaluate_pilot_gate(stage1_pilots)
print_table(stage1_pilot_table["rows"])


In [ ]:
if "stage1_pilot_table" not in globals():
    STAGE1_GATE = None
    print("Stage 1 pilot results are not in this session. Run the stage 1 pilot cell, then rerun this cell.")
else:
    STAGE1_GATE = {
        "evaluated_at": now_iso(),
        "rule": (
            "Full-data stage 1 runs are blocked when any pilot records non-finite values. Loss spikes above "
            "the threshold require ALLOW_STAGE1_FULL_RUN_AFTER_SPIKES to be set after review. A collapsed "
            "pilot is not a block by itself because the raw baseline is expected to be near chance."
        ),
        **stage1_pilot_table,
    }
    save_json("stage1_pilot_gate.json", STAGE1_GATE)

    print("Gate verdict:", STAGE1_GATE["verdict"])
    if STAGE1_GATE["verdict"] == "stop":
        print("Non-finite values were recorded. Investigate training and BatchNorm behaviour before full-data runs.")
    elif STAGE1_GATE["verdict"] == "review_spikes":
        print("Loss spikes were recorded. Review the spike steps and loss values in the pilot records.")
        print("Set ALLOW_STAGE1_FULL_RUN_AFTER_SPIKES = True in the stage 1 configuration cell and rerun this")
        print("cell only if the spikes are transient and harmless.")
    else:
        print("Pilot stability gate passed. Full-data runs are allowed.")


In [ ]:
if "stage1_full_runs" not in globals():
    stage1_full_runs = {}

if STAGE1_GATE is None:
    print("Stage 1 pilot gate is not available in this session. Run the stage 1 pilot and gate cells first.")
elif STAGE1_GATE["verdict"] == "stop" or (
        STAGE1_GATE["verdict"] == "review_spikes" and not ALLOW_STAGE1_FULL_RUN_AFTER_SPIKES):
    print("Stage 1 full-data runs are blocked by the pilot gate. See the gate cell above.")
else:
    print("Stage 1 full run | raw_control")
    record, filename, reused = run_or_reuse_full(
        "stage1_full_", "raw_control", raw_transform, {}, "baseline stride pattern")
    stage1_full_runs["raw_control"] = record
    print(("Reused " if reused else "Saved ") + filename)
    print_table([full_run_row("raw_control", record)])


In [ ]:
if "stage1_full_runs" not in globals():
    stage1_full_runs = {}

if STAGE1_GATE is None:
    print("Stage 1 pilot gate is not available in this session. Run the stage 1 pilot and gate cells first.")
elif STAGE1_GATE["verdict"] == "stop" or (
        STAGE1_GATE["verdict"] == "review_spikes" and not ALLOW_STAGE1_FULL_RUN_AFTER_SPIKES):
    print("Stage 1 full-data runs are blocked by the pilot gate. See the gate cell above.")
else:
    print("Stage 1 full run | residual")
    record, filename, reused = run_or_reuse_full(
        "stage1_full_", "residual", residual_transform, {}, "baseline stride pattern")
    stage1_full_runs["residual"] = record
    print(("Reused " if reused else "Saved ") + filename)
    print_table([full_run_row("residual", record)])


In [ ]:
STAGE1_ARMS_FOR_SUMMARY = {
    "raw_control": (raw_transform, {}, "baseline stride pattern"),
    "residual": (residual_transform, {}, "baseline stride pattern"),
}

stage1_records = {}
for arm_name, (transform, model_kwargs, architecture) in STAGE1_ARMS_FOR_SUMMARY.items():
    match = find_experiment_record(
        ("stage1_full_",), full_run_settings(transform, model_kwargs, architecture))
    if match is not None:
        stage1_records[arm_name] = match["record"]
        print(f"{arm_name}: using {match['filename']}")

stage1_rows = [full_run_row(name, record) for name, record in stage1_records.items()]
if stage1_rows:
    print_table(stage1_rows)

if "raw_control" in stage1_records and "residual" in stage1_records:
    stage1_decision = stage_decision(
        control_record=stage1_records["raw_control"],
        candidate_record=stage1_records["residual"],
        keep_label="residual",
        control_label="raw",
    )
else:
    stage1_decision = {
        "recommendation": "unresolved",
        "gain": None,
        "near_chance_margin": NEAR_CHANCE_MARGIN,
        "keep_threshold": DECISION_MIN_VALIDATION_GAIN,
        "control": None,
        "candidate": None,
    }

STAGE1_SUMMARY = {
    "summarised_at": now_iso(),
    "decision_rule": (
        "Two arms at or below chance make the stage inconclusive. The residual input is kept only when "
        "it is above chance and its selected-checkpoint validation accuracy exceeds the raw control by at "
        f"least {DECISION_MIN_VALIDATION_GAIN}."
    ),
    "decision": stage1_decision,
    "recommendation": stage1_decision["recommendation"],
    "residual_minus_raw_validation_accuracy": stage1_decision["gain"],
    "comparison": stage1_rows,
    "pilot_gate_verdict": None if STAGE1_GATE is None else STAGE1_GATE["verdict"],
}
save_json("stage1_summary.json", STAGE1_SUMMARY)

print("Stage 1 recommendation:", stage1_decision["recommendation"])
if stage1_decision["recommendation"] == "inconclusive_no_learning":
    print("Both arms stayed at chance level, so neither representation is supported by this run.")
    print("Stage 2 stays blocked until the representation, the optimisation, or the data is investigated.")
    print("To run Stage 2 anyway, set STAGE2_INPUT_OVERRIDE and record the reason.")
elif stage1_decision["recommendation"] == "unresolved":
    print("Stage 1 is unresolved. Review the full-run records and the pilot gate before choosing the")
    print("stage 2 input.")


## Stage 2. Delay first downsampling

**Question.** With the input selected in stage 1, does preserving full resolution for one more residual block improve 1-bit detection?

**Change.** `residual_block4` moves from stride 2 to stride 1 and `residual_block5` moves from stride 1 to stride 2. The first reduction happens one block later, and the flattened feature size stays the same.

**Known confound.** Block 5 now changes its channel count while it also down samples, so it gains a 1 by 1 shortcut convolution and a matching BatchNorm. That adds 1,088 learnable parameters that the baseline block 5 does not have. The stage therefore tests a later first reduction together with that projection. Record the parameter difference with the result and do not describe the stage as a stride-only change.

**Method.** The unchanged stride pattern is the control. It reuses the matching stage 1 full run when the settings are identical, which keeps the comparison exact without repeating a finished run. Both arms use the same input transform, split, seeds, batch size, and epoch budget. Shape, memory, and safety checks run before any training.

**Decision.** The delayed pattern is kept only when it is above chance and beats the control by the configured margin. Two chance-level arms are reported as inconclusive.

**Gate.** Training stays blocked until the memory probe passes for both stride patterns at the configured batch size.


In [ ]:
stage1_summary = load_json("stage1_summary.json")
STAGE1_RECOMMENDED_INPUT = stage1_summary["recommendation"]

# Set this to "raw" or "residual" to override the stage 1 recommendation. Leave it empty to use the
# recommendation. An override needs a reason on the next line, because it decides what the stage 2
# comparison is measured against.
STAGE2_INPUT_OVERRIDE = ""
STAGE2_INPUT_OVERRIDE_REASON = ""

# Filled by the memory probe cell. Stage 2 training stays blocked until it is True.
STAGE2_MEMORY_OK = None

if STAGE2_INPUT_OVERRIDE in INPUT_TRANSFORMS:
    STAGE2_INPUT_SELECTED = STAGE2_INPUT_OVERRIDE
elif STAGE1_RECOMMENDED_INPUT in INPUT_TRANSFORMS:
    STAGE2_INPUT_SELECTED = STAGE1_RECOMMENDED_INPUT
else:
    STAGE2_INPUT_SELECTED = None

STAGE2_TRANSFORM = INPUT_TRANSFORMS[STAGE2_INPUT_SELECTED] if STAGE2_INPUT_SELECTED else None

STAGE2_ARMS = {
    "unchanged_stride": {"model_kwargs": {}, "architecture": "baseline stride pattern"},
    "delayed_first_downsampling": {"model_kwargs": {"delay_first_downsampling": True},
                                   "architecture": "delayed first downsampling"},
}

save_json("stage2_configuration.json", {
    "recorded_at": now_iso(),
    "stage1_recommendation": STAGE1_RECOMMENDED_INPUT,
    "stage1_decision": stage1_summary.get("decision"),
    "stage2_input": STAGE2_INPUT_SELECTED,
    "override": STAGE2_INPUT_OVERRIDE,
    "override_reason": STAGE2_INPUT_OVERRIDE_REASON,
    "arms": {name: arm["model_kwargs"] for name, arm in STAGE2_ARMS.items()},
})

print("Stage 1 recommendation:", STAGE1_RECOMMENDED_INPUT)
print("Stage 2 input:", STAGE2_INPUT_SELECTED,
      "->", describe_transform(STAGE2_TRANSFORM) if STAGE2_TRANSFORM else None)
if STAGE2_INPUT_OVERRIDE:
    print("Override in use:", STAGE2_INPUT_OVERRIDE,
          "| reason:", STAGE2_INPUT_OVERRIDE_REASON or "not recorded")
if STAGE2_TRANSFORM is None:
    print("Stage 1 has no usable decision. Set STAGE2_INPUT_OVERRIDE to 'raw' or 'residual' with a")
    print("recorded reason to run stage 2 as an independent downsampling experiment, then rerun this cell.")


In [ ]:
def describe_shapes(model_kwargs):
    seed_everything(SEED)
    model = ResidualSteganalysisCNN(**model_kwargs).to(DEVICE).eval()
    with torch.no_grad():
        tensor = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        shapes = {"input": list(tensor.shape)}
        for index, block in enumerate(model.blocks, start=1):
            tensor = block(tensor)
            shapes[f"residual_block{index}"] = list(tensor.shape)
        flattened = tensor.view(tensor.size(0), -1)
        shapes["flattened_features"] = list(flattened.shape)
        shapes["fc1_in_features"] = int(model.fc1.in_features)
        shapes["logits"] = list(model.fc2(model.relu(model.fc1(flattened))).shape)
    return shapes


def count_parameters(model_kwargs):
    seed_everything(SEED)
    model = ResidualSteganalysisCNN(**model_kwargs)
    return int(sum(parameter.numel() for parameter in model.parameters()))


if STAGE2_TRANSFORM is None:
    print("Stage 2 input is not set. Complete the stage 2 configuration cell first.")
else:
    shape_checks = {
        "baseline stride pattern": describe_shapes({}),
        "delayed first downsampling": describe_shapes({"delay_first_downsampling": True}),
    }
    parameter_counts = {
        "baseline stride pattern": count_parameters({}),
        "delayed first downsampling": count_parameters({"delay_first_downsampling": True}),
    }
    parameter_delta = (
        parameter_counts["delayed first downsampling"] - parameter_counts["baseline stride pattern"]
    )
    save_json("stage2_shape_check.json", {
        "checked_at": now_iso(),
        "input_transform": describe_transform(STAGE2_TRANSFORM),
        "shapes": shape_checks,
        "parameter_counts": parameter_counts,
        "parameter_delta": parameter_delta,
        "parameter_note": (
            "The delayed pattern makes block 5 change channels while it down samples, so block 5 gains a "
            "1 by 1 shortcut convolution and a BatchNorm. The parameter difference belongs to the compared "
            "change and must be reported with the result."
        ),
    })
    print(json.dumps(shape_checks, indent=2))
    print(f"Parameters: baseline {parameter_counts['baseline stride pattern']}, "
          f"delayed {parameter_counts['delayed first downsampling']}, difference {parameter_delta}")

    problems = []
    for name, shapes in shape_checks.items():
        if shapes["flattened_features"][1] != shapes["fc1_in_features"]:
            problems.append(
                f"{name} flattened features {shapes['flattened_features'][1]} do not match "
                f"fc1 input {shapes['fc1_in_features']}"
            )
    if problems:
        raise RuntimeError("Shape check failed: " + "; ".join(problems))
    print("Shape check passed for both stride patterns.")


In [ ]:
def measure_peak_memory(model_kwargs, architecture):
    if not torch.cuda.is_available():
        return {"architecture": architecture, "device": "cpu", "status": "skipped", "peak_bytes": None}
    try:
        seed_everything(SEED)
        model = ResidualSteganalysisCNN(**model_kwargs).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
        torch.cuda.reset_peak_memory_stats()
        images = torch.zeros(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        labels = torch.zeros(BATCH_SIZE, dtype=torch.long, device=DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        peak_bytes = int(torch.cuda.max_memory_allocated())
        del images, labels, loss, model, optimizer, criterion
        torch.cuda.empty_cache()
        return {
            "architecture": architecture,
            "device": torch.cuda.get_device_name(0),
            "probe_batch_size": BATCH_SIZE,
            "status": "ok",
            "peak_bytes": peak_bytes,
            "peak_gigabytes": round(peak_bytes / 1024 ** 3, 3),
        }
    except RuntimeError as error:
        torch.cuda.empty_cache()
        return {
            "architecture": architecture,
            "device": torch.cuda.get_device_name(0),
            "probe_batch_size": BATCH_SIZE,
            "status": "failed",
            "error": str(error)[:400],
        }


if STAGE2_TRANSFORM is None:
    STAGE2_MEMORY_OK = None
    print("Stage 2 input is not set. Complete the stage 2 configuration cell first.")
else:
    memory_probe = [
        measure_peak_memory({}, "baseline stride pattern"),
        measure_peak_memory({"delay_first_downsampling": True}, "delayed first downsampling"),
    ]
    STAGE2_MEMORY_OK = all(probe["status"] == "ok" for probe in memory_probe)
    save_json("stage2_memory_gate.json", {
        "checked_at": now_iso(),
        "batch_size": BATCH_SIZE,
        "ok": STAGE2_MEMORY_OK,
        "action": (
            "Reduce BATCH_SIZE in the configuration cell, rerun this cell, and keep the same batch size "
            "for every arm."
        ),
        "probes": memory_probe,
    })
    print_table(memory_probe, columns=["architecture", "status", "probe_batch_size", "peak_gigabytes"])
    if STAGE2_MEMORY_OK:
        print(f"Both stride patterns fit at batch size {BATCH_SIZE}. Stage 2 training is allowed.")
    else:
        print("At least one stride pattern did not fit. Stage 2 training stays blocked.")
        print("Reduce BATCH_SIZE in the configuration cell, rerun this cell, and keep the same batch size")
        print("for every arm.")


In [ ]:
if STAGE2_TRANSFORM is None:
    print("Stage 2 input is not set. Complete the stage 2 configuration cell first.")
elif STAGE2_MEMORY_OK is not True:
    print("Stage 2 is blocked until the memory probe passes for both stride patterns.")
    print("Run the memory probe cell, reduce BATCH_SIZE if needed, then rerun this cell.")
else:
    stage2_pilots = {}
    for arm_name, arm in STAGE2_ARMS.items():
        print("=" * 72)
        print("Stage 2 paired pilot |", arm_name)
        record, filename, reused = run_or_reuse_pilot(
            arm_name, STAGE2_TRANSFORM, arm["model_kwargs"], arm["architecture"], "stage2_pilot_")
        stage2_pilots[arm_name] = record
        print(("Reused " if reused else "Saved ") + filename)

    stage2_pilot_table = evaluate_pilot_gate(stage2_pilots)
    print_table(stage2_pilot_table["rows"])


In [ ]:
# Set True only after reading the stage 2 pilot gate output and judging the recorded loss spikes
# transient and harmless. Non-finite values block the full runs regardless of this flag.
ALLOW_STAGE2_FULL_RUN_AFTER_SPIKES = False

if "stage2_pilot_table" not in globals():
    STAGE2_GATE = None
    print("Stage 2 pilot results are not in this session. Run the stage 2 pilot cell, then rerun this cell.")
else:
    STAGE2_GATE = {
        "evaluated_at": now_iso(),
        "input_transform": describe_transform(STAGE2_TRANSFORM) if STAGE2_TRANSFORM else None,
        "rule": (
            "Full-data stage 2 runs are blocked when any pilot records non-finite values. Loss spikes above "
            "the threshold require ALLOW_STAGE2_FULL_RUN_AFTER_SPIKES to be set after review. A collapsed "
            "pilot is not a block by itself because near-chance behaviour is expected at these settings."
        ),
        **stage2_pilot_table,
    }
    save_json("stage2_pilot_gate.json", STAGE2_GATE)

    print("Gate verdict:", STAGE2_GATE["verdict"])
    if STAGE2_GATE["verdict"] == "stop":
        print("Non-finite values were recorded. Investigate training and BatchNorm behaviour before full-data runs.")
    elif STAGE2_GATE["verdict"] == "review_spikes":
        print("Loss spikes were recorded. Review the spike steps and loss values in the pilot records.")
        print("Set ALLOW_STAGE2_FULL_RUN_AFTER_SPIKES = True in the cell above and rerun this cell only if")
        print("the spikes are transient and harmless.")
    else:
        print("Pilot stability gate passed. Full-data runs are allowed.")


In [ ]:
if "stage2_full_runs" not in globals():
    stage2_full_runs = {}

if STAGE2_TRANSFORM is None:
    print("Stage 2 input is not set. Complete the stage 2 configuration cell first.")
elif STAGE2_MEMORY_OK is not True:
    print("Stage 2 is blocked until the memory probe passes for both stride patterns.")
    print("Run the memory probe cell, reduce BATCH_SIZE if needed, then rerun this cell.")
elif STAGE2_GATE is None:
    print("Stage 2 pilot gate is not available in this session. Run the stage 2 pilot and gate cells first.")
elif STAGE2_GATE["verdict"] == "stop" or (
        STAGE2_GATE["verdict"] == "review_spikes" and not ALLOW_STAGE2_FULL_RUN_AFTER_SPIKES):
    print("Stage 2 full-data runs are blocked by the pilot gate. See the gate cell above.")
else:
    print("Stage 2 full run | unchanged_stride")
    record, filename, reused = run_or_reuse_full(
        "stage2_full_", "unchanged_stride", STAGE2_TRANSFORM, {}, "baseline stride pattern")
    stage2_full_runs["unchanged_stride"] = record
    print(("Reused " if reused else "Saved ") + filename)
    print_table([full_run_row("unchanged_stride", record)])


In [ ]:
if "stage2_full_runs" not in globals():
    stage2_full_runs = {}

if STAGE2_TRANSFORM is None:
    print("Stage 2 input is not set. Complete the stage 2 configuration cell first.")
elif STAGE2_MEMORY_OK is not True:
    print("Stage 2 is blocked until the memory probe passes for both stride patterns.")
    print("Run the memory probe cell, reduce BATCH_SIZE if needed, then rerun this cell.")
elif STAGE2_GATE is None:
    print("Stage 2 pilot gate is not available in this session. Run the stage 2 pilot and gate cells first.")
elif STAGE2_GATE["verdict"] == "stop" or (
        STAGE2_GATE["verdict"] == "review_spikes" and not ALLOW_STAGE2_FULL_RUN_AFTER_SPIKES):
    print("Stage 2 full-data runs are blocked by the pilot gate. See the gate cell above.")
else:
    print("Stage 2 full run | delayed_first_downsampling")
    record, filename, reused = run_or_reuse_full(
        "stage2_full_", "delayed_first_downsampling", STAGE2_TRANSFORM,
        {"delay_first_downsampling": True}, "delayed first downsampling")
    stage2_full_runs["delayed_first_downsampling"] = record
    print(("Reused " if reused else "Saved ") + filename)
    print_table([full_run_row("delayed_first_downsampling", record)])


In [ ]:
stage2_records = {}
if STAGE2_TRANSFORM is not None:
    for arm_name, arm in STAGE2_ARMS.items():
        match = find_experiment_record(
            ("stage1_full_", "stage2_full_"),
            full_run_settings(STAGE2_TRANSFORM, arm["model_kwargs"], arm["architecture"]),
        )
        if match is not None:
            stage2_records[arm_name] = match["record"]
            print(f"{arm_name}: using {match['filename']}")

stage2_rows = [full_run_row(name, record) for name, record in stage2_records.items()]
if stage2_rows:
    print_table(stage2_rows)

if "unchanged_stride" in stage2_records and "delayed_first_downsampling" in stage2_records:
    stage2_decision = stage_decision(
        control_record=stage2_records["unchanged_stride"],
        candidate_record=stage2_records["delayed_first_downsampling"],
        keep_label="keep_delayed_downsampling",
        control_label="keep_baseline_downsampling",
    )
elif STAGE2_TRANSFORM is None:
    stage2_decision = {"recommendation": "unresolved_input", "gain": None, "control": None, "candidate": None}
else:
    stage2_decision = {"recommendation": "unresolved", "gain": None, "control": None, "candidate": None}

parameter_note = None
if (RESULTS_DIR / "stage2_shape_check.json").is_file():
    shape_record = load_json("stage2_shape_check.json")
    parameter_note = {
        "parameter_counts": shape_record.get("parameter_counts"),
        "parameter_delta": shape_record.get("parameter_delta"),
        "note": shape_record.get("parameter_note"),
    }

STAGE2_SUMMARY = {
    "summarised_at": now_iso(),
    "input_transform": describe_transform(STAGE2_TRANSFORM) if STAGE2_TRANSFORM else None,
    "decision_rule": (
        "Two arms at or below chance make the stage inconclusive. The delayed pattern is kept only when "
        "it is above chance and its selected-checkpoint validation accuracy exceeds the unchanged stride "
        f"control by at least {DECISION_MIN_VALIDATION_GAIN}."
    ),
    "decision": stage2_decision,
    "recommendation": stage2_decision["recommendation"],
    "delayed_minus_unchanged_validation_accuracy": stage2_decision["gain"],
    "comparison": stage2_rows,
    "parameter_confound": parameter_note,
    "confound_statement": (
        "The delayed pattern also adds a 1 by 1 shortcut convolution and a BatchNorm to block 5. The "
        "comparison covers the later first reduction and that projection together."
    ),
    "pilot_gate_verdict": None if STAGE2_GATE is None else STAGE2_GATE["verdict"],
    "memory_gate_ok": STAGE2_MEMORY_OK,
    "peak_memory_probe": (
        load_json("stage2_memory_gate.json")["probes"]
        if (RESULTS_DIR / "stage2_memory_gate.json").is_file()
        else None
    ),
}
save_json("stage2_summary.json", STAGE2_SUMMARY)

print("Stage 2 recommendation:", stage2_decision["recommendation"])
if parameter_note and parameter_note["parameter_delta"] is not None:
    print(f"Reported parameter difference from the stage 2 change: {parameter_note['parameter_delta']}")
if stage2_decision["recommendation"] == "inconclusive_no_learning":
    print("Both stride patterns stayed at chance level, so neither is supported by this run.")
    print("The next stage should target the representation or the optimisation, not the stride pattern.")
elif stage2_decision["recommendation"] == "unresolved":
    print("Stage 2 is unresolved. At least one full-run record is missing. Check the gate outcome and the")
    print("saved records before drawing a conclusion.")


## Review and exit

Stage 1 is planned, not completed, until the matched raw and residual arms have saved results, their pilot gate outcome is recorded, and a validation-based keep, discard, or inconclusive decision is written into `stage1_summary.json`.

Stage 2 is planned, not completed, until the stride-only comparison has passed the shape, parameter, and memory checks, saved comparable results for both stride patterns, and recorded a decision or a clear stop reason.

Runs are durable. Each full arm writes progress and its best weights after every epoch under `v2_results/checkpoints/` and `v2_results/progress/`, and a rerun with identical settings resumes from the saved epoch. An arm that hits a non-finite loss or gradient stops, is saved as invalid, and keeps no checkpoint.

Records are only reused when the stored settings match exactly, including the manifest digests, the pilot source IDs, and the transform signature. Invalid records are never reused.

Neither stage authorises a held-out test run or work on stages 3 to 8. The test split was not loaded anywhere in this notebook, and model selection used validation data only.

After the run.

1. Copy `v2_results/` into `04_Outputs/CV_test/notebooks/` in the vault and keep the notebook alongside it.
2. Record the observed outcome for each stage and the kept, modified, discarded, or inconclusive change.
3. Report the stage 2 parameter difference whenever the delayed downsampling result is reported.
4. Update the project records with the measured result before any stage 3 work begins.
